# 01 — Audio Feature Exploration

This notebook walks through the **MIR feature extraction pipeline** step by step:

1. Load a raw audio file
2. Run the preprocessing pipeline
3. Extract every feature type
4. Visualise each feature with rich plots
5. Compare features across genres

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

from src.mir.preprocessing.pipeline import AudioPreprocessingPipeline, PipelineConfig
from src.mir.features.extractor import FeatureExtractor, FeatureConfig
from src.mir.utils.visualization import (
    plot_mel_spectrogram, plot_waveform, plot_mfcc,
    plot_chroma, plot_feature_dashboard, fig_to_bytes
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

## 1. Generate a Synthetic Test Signal

Using a 440 Hz sine wave so this notebook runs without any real audio files.

In [ ]:
SR = 22050
DURATION = 10.0  # seconds

# Composite signal: 440Hz + 880Hz + some noise
t = np.linspace(0, DURATION, int(SR * DURATION))
y = (0.5 * np.sin(2 * np.pi * 440 * t) +
     0.3 * np.sin(2 * np.pi * 880 * t) +
     0.05 * np.random.randn(len(t))).astype(np.float32)

print(f'Signal: {len(y)} samples @ {SR} Hz = {DURATION:.1f}s')
print(f'Amplitude range: [{y.min():.3f}, {y.max():.3f}]')

## 2. Waveform + Mel Spectrogram

In [ ]:
fig = plot_waveform(y, SR, title='Composite 440+880 Hz Waveform')
plt.show()

fig = plot_mel_spectrogram(y, SR, title='Mel Spectrogram (128 bins)')
plt.show()

## 3. MFCC Extraction

In [ ]:
fig = plot_mfcc(y, SR, n_mfcc=20, title='MFCCs (20 coefficients)')
plt.show()

# Show MFCC statistics
mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20)
print(f'MFCC shape: {mfcc.shape}  (n_mfcc × T_frames)')
print(f'Mean of each MFCC:\n{mfcc.mean(axis=1).round(2)}')

## 4. Chroma Features

In [ ]:
fig = plot_chroma(y, SR, title='Chroma CQT — Pitch Class Energy')
plt.show()

chroma = librosa.feature.chroma_cqt(y=y, sr=SR)
pitch_classes = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
dominant = pitch_classes[chroma.mean(axis=1).argmax()]
print(f'Dominant pitch class: {dominant}')

## 5. Full Feature Extraction

In [ ]:
extractor = FeatureExtractor(FeatureConfig(n_mfcc=40, n_mels=128))
features = extractor.extract(y=y, sr=SR)

print(f'Extraction time : {features.extraction_time_ms:.1f} ms')
print(f'Duration        : {features.duration_seconds:.2f}s')
print(f'Tempo           : {features.tempo["bpm"]:.1f} BPM')
print(f'ZCR mean        : {features.zcr["mean"]:.4f}')
print(f'RMS mean        : {features.rms_energy["mean"]:.4f}')
print(f'Spectral centroid mean: {features.spectral_centroid["mean"]:.1f} Hz')

## 6. Dashboard — All Features at Once

In [ ]:
fig = plot_feature_dashboard(y, SR, title='MIR Feature Dashboard')
plt.show()

## 7. Flat Feature Vector for Classical ML

In [ ]:
vec = features.to_flat_vector()
print(f'Flat vector shape : {vec.shape}')
print(f'dtype             : {vec.dtype}')
print(f'Any NaN?          : {np.isnan(vec).any()}')
print(f'Sample values     : {vec[:10].round(4)}')